# ViralCut AI — Colab Runner (Simple)

Bas ye 2 steps karo:
1. Upar **Runtime → Change runtime type → GPU (T4)** select karo
2. **Runtime → Run all** dabao (ya har cell ko ▶️ se chalao)

Koi code/log screen par nahi dikhega — sirf **aakhir mein tumhara link** aayega. Pehli baar chalne mein 3-5 minute lagte hain (model + fonts download hote hain), thoda sabar karo.

In [ ]:
%%capture
# Sab kuch is cell ke andar chup-chaap hota hai — koi output screen par nahi aata

import subprocess, time, re, os

# 1) Repo clone
if not os.path.exists("ViralCut-AI"):
    subprocess.run(["git", "clone", "https://github.com/itxunknown39-web/ViralCut-AI.git"])
os.chdir("ViralCut-AI")

# 2) System deps + Python requirements
subprocess.run(["apt-get", "-qq", "update"])
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "requests"])

# 3) cloudflared (no login, no token)
if not os.path.exists("cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared",
    ])
    subprocess.run(["chmod", "+x", "cloudflared"])

# 4) Server start
server_log = open("server.log", "w")
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
)

# 5) Health-check loop (isse Error 1033 nahi aata — tunnel server se pehle nahi banta)
import requests
ready = False
for attempt in range(90):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

# 6) Tunnel start (sirf jab server ready ho)
public_url = None
if ready:
    tunnel_log = open("tunnel.log", "w")
    tunnel_proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=tunnel_log, stderr=subprocess.STDOUT,
    )
    for attempt in range(30):
        time.sleep(2)
        with open("tunnel.log") as f:
            text = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if match:
            public_url = match.group(0)
            break

In [ ]:
# Sirf ye cell result dikhayega
if public_url:
    print("Tumhara ViralCut AI yahan open hai:\n")
    print(public_url)
elif not ready:
    print("Server start nahi ho saka. server.log ke aakhri 40 lines:\n")
    !tail -n 40 server.log
else:
    print("Link abhi nahi bana. tunnel.log ke aakhri 40 lines:\n")
    !tail -n 40 tunnel.log

---
Session khatam hone par dobara link chahiye ho to bas dono cells upar se phir chala do.